In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/competitions/predict-survival-of-the-passengers/maritime_train.csv
/kaggle/input/competitions/predict-survival-of-the-passengers/maritime_sample_submission.csv
/kaggle/input/competitions/predict-survival-of-the-passengers/maritime_test.csv


In [2]:
mss=pd.read_csv('/kaggle/input/competitions/predict-survival-of-the-passengers/maritime_sample_submission.csv')
mtest=pd.read_csv('/kaggle/input/competitions/predict-survival-of-the-passengers/maritime_test.csv')
mtrain=pd.read_csv('/kaggle/input/competitions/predict-survival-of-the-passengers/maritime_train.csv')

In [3]:
y=mtrain['Outcome']
mtrain.drop(['PassengerId','PassengerName','Berth','Outcome'],axis=1,inplace=True)
x=mtrain.copy()

In [4]:
from sklearn.model_selection import train_test_split
x_train,x_test,y_train,y_test=train_test_split(x,y,test_size=0.2,random_state=42)

In [5]:
age_mean=x_train['Age'].mean()
x_train['Age']=x_train['Age'].fillna(age_mean)
x_test['Age']=x_test['Age'].fillna(age_mean)

In [6]:
import re

def get_prefix(ticket):
  ticket=ticket.strip()
  prefix=re.sub(r'[^A-Za-z/]','',ticket)
  if prefix == '':
    return 'Numeric'
  return prefix.upper()  
x_train['TicketPrefix']=x_train['CLass'].apply(get_prefix)  
x_test['TicketPrefix']=x_test['CLass'].apply(get_prefix)

In [7]:
x_train.drop('CLass',axis=1,inplace=True)
x_test.drop('CLass',axis=1,inplace=True)

In [8]:
cat_ole=['TicketPrefix']
cat_ohe=['Gender','BoardingPort','Title','TicketPrefix']
num_cols=['TicketTier','Age','RelativesAboard','ParentsChildren','TicketCost','FamilySize','FarePerPerson']

In [9]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder,OrdinalEncoder,StandardScaler,LabelEncoder

In [10]:
num_t=Pipeline([('scaler',StandardScaler())])
cat_ohe_t=Pipeline([('onehot',OneHotEncoder(handle_unknown='ignore'))])
pp=ColumnTransformer(transformers=[('1',num_t,num_cols),('2',cat_ohe_t,cat_ohe)])
from sklearn.linear_model import LogisticRegression
x_train_pp=pp.fit_transform(x_train)
x_test_pp=pp.transform(x_test)

In [11]:
from sklearn.metrics import accuracy_score
model=LogisticRegression(
    penalty='l1',
    solver='saga',
    max_iter=1000,random_state=42
)
model.fit(x_train_pp,y_train)
y_pred=model.predict(x_test_pp)
acc=accuracy_score(y_test,y_pred)
print(acc)

0.8321678321678322


/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_sag.py:348: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(


In [12]:
from sklearn.linear_model import RidgeClassifier
from sklearn.metrics import accuracy_score

model = RidgeClassifier()
model.fit(x_train_pp,y_train)
y_pred=model.predict(x_test_pp)
acc=accuracy_score(y_test,y_pred)
print(acc)


0.8251748251748252


In [13]:
from sklearn.ensemble import RandomForestClassifier
model1=RandomForestClassifier(random_state=42)
model1.fit(x_train_pp,y_train)
y_pred=model1.predict(x_test_pp)
acc=accuracy_score(y_test,y_pred)
print(acc)

0.8251748251748252


In [14]:
from xgboost import XGBClassifier

model2 = XGBClassifier(random_state=42)
model2.fit(x_train_pp,y_train)
y_pred=model2.predict(x_test_pp)
acc=accuracy_score(y_test,y_pred)
print(acc)

0.8111888111888111


In [15]:

age_mean1=x['Age'].mean()
x['Age']=x['Age'].fillna(age_mean1)
x['TicketPrefix']=x['CLass'].apply(get_prefix)
x.drop('CLass',axis=1,inplace=True)

x_pp=pp.fit_transform(x)

In [16]:

model=LogisticRegression(
    penalty='l1',
    solver='saga',
    max_iter=1000,random_state=42
)
model.fit(x_pp,y)

/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_sag.py:348: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(


LogisticRegression(max_iter=1000, penalty='l1', random_state=42, solver='saga')

In [17]:

mtest['Age']=mtest['Age'].fillna(age_mean1)
mtest['TicketPrefix']=mtest['CLass'].apply(get_prefix)
mtest.drop('CLass',axis=1,inplace=True)

test_pp=pp.transform(mtest)

In [18]:
k=model.predict(test_pp)

In [19]:
submission=mss.copy()
submission.head()
submission['Outcome']=k

In [20]:
submission.to_csv("/kaggle/working/submission.csv", index=False)